## **Greedy Decoding**

In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

input_txt = "Transformer are the"
input_ids = tokenizer(input_txt, return_tensors = 'pt')["input_ids"].to(device)
iterations = []
n_steps = 8
choices_per_step = 5

with torch.no_grad():
  for _ in range(n_steps):
    iteration = dict()
    iteration["Input"] = tokenizer.decode(input_ids[0])
    output = model(input_ids = input_ids)
    # select logits of the first batch and the last token and apply softmax
    next_token_logits = output.logits[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits, dim = -1)
    sorted_ids = torch.argsort(next_token_probs, dim = -1, descending = True)
    # store the tokens with highest probabilities
    for choice_idx in range(choices_per_step):
      token_id = sorted_ids[choice_idx]
      token_prob = next_token_probs[token_id].cpu().numpy()
      token_choice = (
        f"{tokenizer.decode(token_id)} ({100 * token_prob:.2f}%)"
      )
      iteration[f"Choice {choice_idx + 1}"] = token_choice

    # append predicted next token to input
    input_ids = torch.cat([input_ids, sorted_ids[None, 0, None]], dim = -1)
    iterations.append(iteration)

pd.DataFrame(iterations)

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

,Input,Choice 1,Choice 2,Choice 3,Choice 4,Choice 5
0,Transformer are the,most (9.20%),only (4.13%),same (4.07%),core (3.30%),ones (2.97%)
1,Transformer are the most,common (19.33%),important (11.11%),popular (5.48%),powerful (5.08%),commonly (4.10%)
2,Transformer are the most common,types (7.73%),type (5.39%),. (4.97%),", (4.80%)",and (4.66%)
3,Transformer are the most common types,of (76.19%),. (5.01%),used (3.97%),", (3.34%)",in (1.40%)
4,Transformer are the most common types of,mod (5.03%),mods (4.12%),transformer (3.07%),trans (2.15%),Transformers (1.59%)
5,Transformer are the most common types of mod,. (13.87%),ded (11.06%),", (7.68%)",in (6.82%),ulators (6.42%)
6,Transformer are the most common types of mod.,They (21.68%),\n (12.27%),The (5.73%),There (3.06%),Most (2.65%)
7,Transformer are the most common types of mod. ...,are (30.61%),can (6.57%),'re (5.00%),add (3.25%),have (3.23%)


#### using hugging face `generate()` function

In [5]:
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output = model.generate(input_ids, max_new_tokens=n_steps, do_sample=False)
print(tokenizer.decode(output[0]))

Transformer are the most common types of mod. They are


let's generate something interesting more big

In [9]:
max_length = 128
input_txt = """In a shocking finding, scientist discovered \
a herd of unicorns living in a remote, previously unexplored \
valley, in the Andes Mountains. Even more surprising to the \
researchers was the fact that the unicorns spoke perfect English.\n\n
"""
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_greedy = model.generate(input_ids, max_length=max_length,
 do_sample=False)
print(tokenizer.decode(output_greedy[0]))

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The researchers, from the University of California, Davis, and the University of Colorado, Boulder, were conducting a study on the Andean cloud forest, which is home to the rare species of cloud forest trees.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.


The researchers were surprised to find that the unicorns were able
